In [ ]:
import pandas as pd
import seaborn as sns
sns.set_style("whitegrid")
import numpy as np 
import scipy.stats as stats
from collections import Counter
import matplotlib.pyplot as plt
import umap
import matplotlib
import mygene 
%matplotlib inline
import pickle
import sklearn
import random
import scanpy as sc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans
from xgboost import XGBClassifier
# import sentence_transformers
plt.style.use('ggplot')
#plt.style.use('seaborn-v0_8-dark-palette')
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('retina')
import openai
# remember to set your open AI API key!
openai.api_key = 
np.random.seed(202310)
# use hnswlib for NN classification
try:
    import hnswlib
    hnswlib_imported = True
except ImportError:
    hnswlib_imported = False
    print("hnswlib not installed! We highly recommend installing it for fast similarity search.")
    print("To install it, run: pip install hnswlib")
from scipy.stats import mode, zscore
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
import torch
import json

Define a new FX for LLM embedding generation

In [ ]:
def get_seq_embed_gpt(X, gene_names, prompt_prefix="", trunc_index = None):
    n_genes = X.shape[1]
    if trunc_index is not None and not isinstance(trunc_index, int):
        raise Exception('trunc_index must be None or an integer!')
    elif isinstance(trunc_index, int) and trunc_index>=n_genes:
        raise Exception('trunc_index must be smaller than the number of genes in the dataset')
    get_test_array = []
    for cell in (X):
        cell = cell.toarray().flatten() if hasattr(cell, "toarray") else cell.flatten()
        zero_indices = (np.where(cell==0)[0])
        gene_indices = np.argsort(cell)[::-1]
        filtered_genes = gene_indices[~np.isin(gene_indices, list(zero_indices))]
        if trunc_index is not None:
            get_test_array.append(np.array(gene_names[filtered_genes])[0:trunc_index]) 
        else:
            get_test_array.append(np.array(gene_names[filtered_genes])) 
    get_test_array_seq = [prompt_prefix+' '.join(x) for x in get_test_array]
    return(get_test_array_seq)

Access to OpenAI

In [ ]:
def get_gpt_embedding(text, model="text-embedding-ada-002"):
    text = text.replace("\n", " ")
    return np.array(openai.Embedding.create(input = [text], model=model)['data'][0]['embedding'])

In [ ]:
MN_adata = sc.read_h5ad

import anndata as ad
MN_adata = ad.AnnData(X=MN_adata.raw.X.toarray(),
                      obs=MN_adata.obs.copy(),
                      var=MN_adata.raw.var.copy())

Embedding generation

In [ ]:
N_TRUNC_GENE = 5
MN_data = get_seq_embed_gpt(
    MN_adata.X,
    np.array(MN_adata.var.index), 
    prompt_prefix = 'A cell with genes ranked by expression: ',trunc_index=N_TRUNC_GENE
)

In [ ]:
# Import dictionary for OpenAI to understand
with open(“#file”, 'rb') as handle:
    gene_dict = json.load(handle)

with open(“#file”, "rb") as f:
    gene_dict = pickle.load(f)


In [ ]:
# Create a new dictionary with all keys capitalized
gene_dict_upper = {gene.upper(): desc for gene, desc in gene_dict.items()}
for gene, desc in gene_dict_upper.items():
    if desc == "":
        gene_dict_upper[gene] = "This gene has no NCBI description."

Gene dictionary

In [ ]:
top_genes = MN_data[0].split(": ")[1].split()

gene_descriptions = {}
missing_genes = []

# Convert gene names to uppercase for comparison
top_genes_upper = [gene.upper() for gene in top_genes]

# Populate the gene_descriptions dictionary
for gene in top_genes:
    if gene in gene_dict_upper:
        gene_descriptions[gene] = gene_dict_upper[gene]
    else:
        missing_genes.append(gene)

print(f"Descriptions found for {len(gene_descriptions)} genes. {len(missing_genes)} genes missing.")

In [ ]:
import numpy as np

cell_idx = 2  
# Extract expression values for the cell
expr_values = MN_adata.X[cell_idx, :].toarray().flatten() if hasattr(MN_adata.X, 'toarray') else MN_adata.X[cell_idx, :]

# Identify the top 20 highest expression genes
top20_idx = np.argsort(expr_values)[::-1][:20]
top20_genes = MN_adata.var.index[top20_idx]

print("Cell", cell_idx, "Top 20 genes:", top20_genes)

Embedding generation

In [ ]:
checkpoint_embeddings_140 = list(np.load("/Users/checkpoint_embeddings_140.npy"))

N_TRUNC_GENE = 20
MN_data = get_seq_embed_gpt( 
    MN_adata.X, 
    np.array(MN_adata.var.index), 
    prompt_prefix='A cell with genes ranked by expression: ', 
    trunc_index=N_TRUNC_GENE 
)

MN_adata.var.index = MN_adata.var.index.str.upper() 

# Initialize an empty list to store the gene embeddings for all cells
all_normalized_embeddings = []
# all_normalized_embeddings = checkpoint_embeddings_X

# Process each cell
for cell_idx in tqdm(range(140, len(MN_data)), desc="Processing cells"): 
    top_genes = MN_data[cell_idx].split(": ")[1].split() 

    gene_descriptions = {}
    missing_genes = []
    
    # Convert gene names to uppercase for comparison
    top_genes_upper = [gene.upper() for gene in top_genes] 
    
    # Extract descriptions for the genes
    for gene in top_genes_upper:
        if gene in gene_dict_upper:
            gene_descriptions[gene] = gene_dict_upper[gene]
        else:
            missing_genes.append(gene)
    
    # Generate embeddings
    embeddings = {}
    for gene, desc in gene_descriptions.items():
        embedding = get_gpt_embedding(desc)
        embeddings[gene] = embedding
    
    # Extract expression levels
    expression_levels = {} 
    for gene in embeddings.keys(): 
        if gene in MN_adata.var.index:
            gene_index = np.where(MN_adata.var.index == gene)[0][0]
            expression_levels[gene] = MN_adata.X[cell_idx, gene_index] 
        else:
            print(f"Warning: Gene {gene} not found in MN_adata.")
    
    # Convert to NumPy arrays 
    expression_values = np.array(list(expression_levels.values())) 
    embedding_values = np.array([embeddings[gene] for gene in expression_levels])

    # Perform weighted normalization
    if np.sum(expression_values) == 0:
        print(f"Cell {cell_idx}: All expression levels are zero. Returning zero vector.")
        normalized_embedding = np.zeros(embedding_values.shape[1])
    else:
        normalized_embedding = np.sum(embedding_values * expression_values[:, np.newaxis], axis=0) / np.sum(expression_values)
    all_normalized_embeddings.append(normalized_embedding)

    # Save checkpoint every 20 cells
    if cell_idx % 20 == 0:
        np.save(f"/Users/checkpoint_embeddings_{cell_idx}.npy", np.array(all_normalized_embeddings))

# Convert to NumPy array
all_normalized_embeddings = np.array(all_normalized_embeddings)
print(f"Generated normalized embeddings with shape: {all_normalized_embeddings.shape}")


Confusion matrix

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
MN_embedding = np.load()

In [ ]:
cell_type_map = {
    "aff": "a fast-fatigue",
    "afr": "a fast-fatigue resistant",
    "asf": "a slow-firing",
    "g": "Y"
    }

MN_adata.obs["cell_types"] = MN_adata.obs["cell_type"].map(cell_type_map)

x = MN_embedding
y = MN_adata.obs["cell_type"].values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=1)
# 20% sample distribute to test data, 80% sample distribute to train data.

knn = KNeighborsClassifier(n_neighbors=10, metric="cosine")
knn.fit(x_train, y_train) 

y_pred = knn.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')  # 'weighted' accounts for class imbalance
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print(f"F1 Score (weighted): {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False

In [ ]:
label_dict = {
    "afr": "α FR MN",
    "aff": "α FF MN",
    "asf": "α SF MN",
    "g": "γ MN"
}

In [ ]:
label_order = np.unique(np.concatenate([y_test, y_pred]))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=label_order)
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)

# Define class labels dynamically from y_test
labels = np.unique(np.concatenate([y_test, y_pred]))
pretty_labels = [label_dict[l] for l in label_order]


# Plot the normalized confusion matrix
plt.figure(figsize=(9, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", xticklabels=pretty_labels, yticklabels=pretty_labels, cbar_kws={"shrink": 0.85}, vmin=0, vmax=1, annot_kws={"size":15, "weight":"normal"})
ax = plt.gca()

ax.set_xticklabels(pretty_labels, fontsize=20, fontweight="normal", fontfamily="Arial", rotation=0, color="black")
ax.set_yticklabels(pretty_labels, fontsize=20, fontweight="normal", fontfamily="Arial", rotation=0, color="black")
for spine in ax.spines.values(): spine.set_visible(True), spine.set_edgecolor("black"), spine.set_linewidth(1)
cbar = ax.collections[0].colorbar
ticks = list(cbar.get_ticks())
ticks = sorted(set(ticks + [0,1]))

cbar.set_ticks(ticks)
cbar.outline.set_edgecolor("black")
cbar.outline.set_linewidth(1)
cbar.ax.tick_params(labelsize=15, labelfontfamily="Arial", color="black")
for label in cbar.ax.get_yticklabels():label.set_fontweight("normal")

plt.xlabel("Predicted Labels", fontsize=20, fontweight="normal", fontfamily="Arial", color="black")
plt.ylabel("True Labels",fontsize=20, fontweight="normal", fontfamily="Arial", color="black")
plt.title(f"Normalized Confusion Matrix for KNN Classifier\n With Embedding, Without Contrastive Learning \nModel: text-embedding-ada-002\nAccuracy: {accuracy:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}",
          fontsize=17, fontweight="normal", fontfamily="Arial", color="black")
plt.tight_layout()

plt.show()

# Print classification report
labels = np.unique(np.concatenate([y_test, y_pred]))
print("Classification Report:")
print(classification_report(y_test, y_pred, labels=label_order, target_names=label_dict))

Umap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sklearn.manifold import TSNE
import scanpy as sc
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
from matplotlib.ticker import MultipleLocator

In [ ]:
# Upload embedding processed npy file
MN_embedding = np.load("")

In [ ]:
label_dict_umap = {
    "afr": "α FR MN",
    "aff": "α FF MN",
    "asf": "α SF MN",
    "g": "γ MN"
}

In [ ]:
# Load your embeddings and labels
labels = MN_adata.obs['cell_type'].values  

# UMAP
reducer_umap = umap.UMAP(random_state=1)
embedding_umap = reducer_umap.fit_transform(MN_embedding)

plt.rcParams["font.family"] = "Arial"
plt.figure(figsize=(10, 8))
plt.style.use('default')

# color_map = {"a": "#ff7f0e","g": "#1f77b4"}
              
for label in np.unique(labels):
    idx = labels == label
    plt.scatter(
        embedding_umap[idx, 0], 
        embedding_umap[idx, 1], 
        label=label_dict_umap.get(str(label), f"Cluster{label}"),alpha=0.8, zorder=2)
        # color=color_map.get(label))
        #label=label, s=15, alpha=0.8, zorder=2)

plt.grid(True, linestyle='--', linewidth=1.0, alpha=1.0, zorder=0)

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(2.5))   
ax.yaxis.set_major_locator(MultipleLocator(2.5))
ax.tick_params(axis='both', labelsize=20)
for spine in ax.spines.values():
    spine.set_linewidth(1)
    spine.set_color("black")
for label in ax.get_xticklabels():
    label.set_fontfamily("Arial")
for label in ax.get_yticklabels():
    label.set_fontfamily("Arial")


# Model: text-embedding-002
plt.title("UMAP of KNN Classifier\nWith Embedding, Without Contrastive Learning\nModel: text-embedding-ada-002",fontsize=20, fontfamily="Arial")
plt.legend(loc="upper right", bbox_to_anchor=(1, 1),prop={"family": "Arial", "size":15})
plt.xlabel("UMAP1",fontsize=20, fontfamily="Arial")
plt.ylabel("UMAP2",fontsize=20)
plt.grid(True)
plt.show()

sil_score = silhouette_score(MN_embedding, labels, metric='cosine', random_state =1)
print(f"Silhouette Score (LLM embeddings): {sil_score:.4f}")
ch_score = calinski_harabasz_score(MN_embedding, labels)
print(f"Calinski-Harabasz Score (LLM embeddings): {ch_score:.2f}")
db_score = davies_bouldin_score(MN_embedding, labels)
print(f"Davies-Bouldin Score (LLM embeddings): {db_score:.2f}")